# Topic Modeling e Caratterizzazione con BERTopic e Jina Embeddings v3

Questo notebook implementa una pipeline avanzata per il clustering e la caratterizzazione dei report di insicurezza alimentare (IPC) usando **BERTopic** e **Jina Embeddings v3**.

L'obiettivo è raggruppare i report in base alle cause scatenanti (conflitti, siccità, crisi economiche) e alle condizioni di insicurezza alimentare, superando i limiti del precedente approccio (K-Means standard + TF-IDF esterno).

### Miglioramenti chiave rispetto al primo approccio:
1. **Gestione del Contesto Lungo**: Sostituzione di `instructor-large` (finestra di 512 token, insufficiente per report lunghi fino a 1800 token) con **`jinaai/jina-embeddings-v3`**, che offre una finestra di contesto nativa di **8192 token**.
2. **Adapter Specifico per il Clustering**: Sfruttamento del LoRA adapter `task="separation"` di Jina v3, addestrato specificamente per ottimizzare la separazione e il raggruppamento nello spazio vettoriale.
3. **c-TF-IDF Nativo**: Integrazione diretta del calcolo TF-IDF per classe/cluster all'interno di BERTopic per caratterizzare in modo ottimale le parole chiave di ciascun gruppo.
4. **Visualizzazioni Avanzate di BERTopic**: Utilizzo delle funzionalità grafiche interattive native di BERTopic per esplorare la mappa dei topic, le gerarchie, e i bar chart dei termini più rilevanti.

## 1. Setup dell'Ambiente e Importazione Librerie

In [ ]:
# 1.1 Monta Google Drive per accedere ai file dei report
from google.colab import drive
import os

drive.mount('/content/drive')

# Definisci i percorsi principali su Google Drive
DRIVE_PATH = "/content/drive/MyDrive/HERO/"
TESTO_DIR = os.path.join(DRIVE_PATH, "key_results_testi")
EMBEDDINGS_FILE = os.path.join(DRIVE_PATH, "embeddings_jina_v3.npy")
VISUALIZZAZIONI_DIR = os.path.join(DRIVE_PATH, "visualizzazioni_bertopic")

print(f"Directory dei report impostata su: {TESTO_DIR}")
if os.path.exists(TESTO_DIR):
    print(f"Trovati {len(os.listdir(TESTO_DIR))} elementi nella cartella.")
else:
    print("ATTENZIONE: Il percorso specificato non esiste. Verifica che Google Drive sia montato correttamente.")

In [ ]:
# 1.2 Installazione pacchetti necessari su Google Colab
# Installiamo bertopic, einops (richiesto da Jina v3) e aggiorniamo sentence-transformers e transformers compatibili
!pip install sentence-transformers umap-learn hdbscan plotly pandas numpy scikit-learn nltk bertopic einops "transformers<5.0.0" -q
print("Librerie installate correttamente!")

In [ ]:
# 1.3 Importazione delle librerie
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.stats as stats
import torch
import nltk
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords

# Per BERTopic e i suoi componenti
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from umap import UMAP
from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import CountVectorizer

# Impostazione seed per riproducibilità
np.random.seed(42)

# Download delle risorse NLTK necessarie
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)
nltk.download('stopwords', quiet=True)

print("Librerie e risorse importate con successo.")

## 2. Caricamento e Pulizia dei Dati

Leggiamo i 497 report testuali. Estraiamo Paese e Periodo dal nome del file, dopodiché puliamo il testo utile eliminando la testata dei metadati (tutto ciò che precede la riga divisoria con `========================================`).

In [ ]:
def estrai_metadata(nome_file):
    base = nome_file.replace("_KeyResults.txt", "")
    
    # Cerca il pattern del periodo: Mese_Anno_-_Mese_Anno
    periodo_match = re.search(r'([A-Za-z]{3}_\d{4})_-_([A-Za-z]{3}_\d{4})', base)
    
    if periodo_match:
        inizio_periodo = periodo_match.group(1).replace('_', ' ')
        fine_periodo = periodo_match.group(2).replace('_', ' ')
        periodo = f"{inizio_periodo} / {fine_periodo}"
    else:
        inizio_periodo = None
        fine_periodo = None
        periodo = base.replace('_', ' ')
        
    # Paese: tutto quello che precede il primo mese o anno
    paese_match = re.match(r'^(.+?)_[A-Za-z]{3}_\d{4}', base)
    paese = paese_match.group(1).replace('_', ' ') if paese_match else base.replace('_', ' ')
    
    return {
        "paese": paese,
        "periodo": periodo,
        "inizio_periodo": inizio_periodo,
        "fine_periodo": fine_periodo,
        "nome_file": nome_file
    }

files = [f for f in os.listdir(TESTO_DIR) if f.endswith(".txt")]
print(f"Trovati {len(files)} file .txt da elaborare.")

testi_puliti = []
metadata_list = []

for nome_file in files:
    path = os.path.join(TESTO_DIR, nome_file)
    with open(path, "r", encoding="utf-8") as f:
        testo_completo = f.read()
    
    # Rimuovi l'intestazione dei metadati (tutto ciò che precede la riga divisoria di '=' o '-')
    testo_pulito = re.sub(r'^.*?={20,}\n*', '', testo_completo, flags=re.DOTALL).strip()
    
    if not testo_pulito:
        print(f"Attenzione: il file {nome_file} è vuoto dopo la pulizia.")
        continue
        
    testi_puliti.append(testo_pulito)
    metadata_list.append(estrai_metadata(nome_file))

# Creazione del DataFrame
df = pd.DataFrame(metadata_list)
df["testo"] = testi_puliti
print(f"Caricati con successo {len(df)} report nel DataFrame.")
df.head()

## 3. Generazione Embedding a Lungo Contesto (Jina Embeddings v3)

Usiamo il modello **`jinaai/jina-embeddings-v3`**. Questo modello permette di elaborare testi fino a **8192 token** senza alcuna troncazione.
Impostiamo inoltre il parametro `task="separation"` per istruire il modello a mappare gli embedding in modo da massimizzare la separazione geometrica utile per il clustering.

Nota: Salviamo gli embedding calcolati su file `.npy` su Drive per velocizzare le esecuzioni successive.

In [ ]:
MODEL_NAME = "jinaai/jina-embeddings-v3"

if os.path.exists(EMBEDDINGS_FILE):
    print(f"Caricamento degli embedding pre-calcolati da {EMBEDDINGS_FILE}...")
    embeddings = np.load(EMBEDDINGS_FILE)
    print(f"Embedding caricati! Shape: {embeddings.shape}")
else:
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Caricamento modello {MODEL_NAME} su dispositivo: {device}...")
    
    # Carichiamo il modello usando trust_remote_code=True per supportare l'architettura Jina v3
    model = SentenceTransformer(MODEL_NAME, device=device, trust_remote_code=True)
    model.max_seq_length = 8192
    
    print("Generazione degli embedding in corso (circa 1-3 minuti su GPU Colab)...")
    # Calcoliamo gli embedding per il task di separazione/clustering
    embeddings = model.encode(
        df["testo"].tolist(),
        task="separation",
        show_progress_bar=True
    )
    
    # Salvataggio su Google Drive
    np.save(EMBEDDINGS_FILE, embeddings)
    print(f"Embedding generati e salvati in {EMBEDDINGS_FILE}. Shape: {embeddings.shape}")

## 4. Configurazione dei Componenti di BERTopic

Configuriamo individualmente i singoli componenti di BERTopic:

1. **UMAP**: Riduzione a 5 dimensioni usando la metrica del coseno per preservare la similarità semantica prima del clustering.
2. **Clustering (K-Means o HDBSCAN)**: 
   - Nel primo approccio K-Means con $K=5$ ha fornito una buona separazione semantica.
   - BERTopic permette di usare **K-Means** al posto di HDBSCAN semplicemente passandolo al costruttore. Questo ci consente di confrontare direttamente i risultati a parità di cluster ($K=5$) o di lasciare a HDBSCAN il compito di trovare cluster densi flessibili.
   - Configureremo una variabile per alternare facilmente tra i due approcci.
3. **CountVectorizer**: Pipeline di tokenizzazione e lemmatizzazione customizzata per rimuovere stop words, nomi dei paesi e termini generici dei report, così da isolare esclusivamente i veri driver della crisi alimentare.

In [ ]:
# 4.1 Inizializzazione UMAP per riduzione dimensionale
umap_model = UMAP(
    n_neighbors=15,
    min_dist=0.0,
    n_components=5,
    metric="cosine",
    random_state=42
)

# 4.2 Selezione del modello di clustering
# Imposta 'kmeans' per replicare la configurazione a K cluster definiti o 'hdbscan' per quella basata sulla densità
CLUSTERING_METHOD = "kmeans"  # Opzioni: "kmeans" o "hdbscan"
K_SCELTO = 5  # Numero di cluster per K-Means

if CLUSTERING_METHOD == "kmeans":
    print(f"Configurazione clustering: K-Means con K={K_SCELTO}")
    cluster_model = KMeans(n_clusters=K_SCELTO, random_state=42, n_init=10)
else:
    import hdbscan
    print("Configurazione clustering: HDBSCAN per rilevamento densità e outlier")
    cluster_model = hdbscan.HDBSCAN(
        min_cluster_size=15,
        min_samples=5,
        metric="euclidean",
        prediction_data=True
    )

In [ ]:
# 4.3 Setup Tokenizzatore e Lemmatizzatore Custom per il Vectorizer di BERTopic
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

# Escludiamo i nomi dei Paesi per evitare bias geografici nei topic
nomi_paesi = set(df["paese"].str.lower().unique())
stop_words.update(nomi_paesi)

# Aggiungiamo termini tecnici e ricorrenti della classificazione IPC
termini_generici = [
    "report", "key", "results", "analysis", "ipc", "period", "country", "province", 
    "provinces", "district", "districts", "million", "people", "population", "acute", 
    "food", "security", "insecurity", "phase", "minimal", "stressed", "crisis", 
    "emergency", "famine", "classified", "situation", "affected", "overall", "total",
    "compared", "projected", "period", "household", "households", "number", "data",
    "level", "levels", "number", "numbers", "area", "areas", "high", "highest"
]
stop_words.update(termini_generics)

def tokenize_and_lemmatize(text):
    # Rimuove numeri e caratteri non alfabetici
    text_pulito = re.sub(r'[^a-zA-Z\s]', '', text)
    tokens = nltk.word_tokenize(text_pulito.lower())
    
    # Lemmatizza e filtra stop words e parole corte
    tokens_lemmatizzati = [
        lemmatizer.lemmatize(t) for t in tokens
        if t not in stop_words and len(t) > 2
    ]
    return tokens_lemmatizzati

# Definiamo il CountVectorizer che BERTopic userà per estrarre le parole rappresentative
vectorizer_model = CountVectorizer(tokenizer=tokenize_and_lemmatize, stop_words=None)

## 5. Addestramento del Modello BERTopic

Creiamo e addestriamo il modello BERTopic passando gli embedding di Jina v3 pre-calcolati. Questo ci assicura che il modello usi lo spazio semantico a lungo contesto da noi definito.

In [ ]:
print("Inizializzazione BERTopic...")
topic_model = BERTopic(
    umap_model=umap_model,
    hdbscan_model=cluster_model,
    vectorizer_model=vectorizer_model,
    calculate_probabilities=True,
    verbose=True
)

print("Fitting del modello BERTopic sui report...")
topics, probs = topic_model.fit_transform(df["testo"].tolist(), embeddings)

# Aggiungiamo i topic generati al DataFrame per le analisi successive
df["topic"] = topics

print("Fitting completato!")

## 6. Interpretazione e Caratterizzazione dei Cluster (Topic)

Vediamo la tabella riassuntiva dei topic estratti. La colonna `Count` indica quanti report appartengono a quel topic, mentre `Name` mostra i termini più significativi calcolati con il c-TF-IDF.

In [ ]:
# Mostra le informazioni generali dei topic estratti
info_topics = topic_model.get_topic_info()
info_topics

In [ ]:
# Stampa delle parole chiave dettagliate per ciascun topic
print("============================================================")
print("CARATTERIZZAZIONE DEI TOPIC (c-TF-IDF)")
print("============================================================")

for topic_id in sorted(df["topic"].unique()):
    if topic_id == -1:
        name = "Outliers / Rumore"
    else:
        # Ottieni le parole chiave del topic
        words = [w[0] for w in topic_model.get_topic(topic_id)[:10]]
        name = f"Topic {topic_id}: {', '.join(words)}"
        
    count = len(df[df["topic"] == topic_id])
    print(f"\n👉 {name} ({count} report)")
    print("-" * 60)
    
    # Mostriamo un estratto del report più rappresentativo per questo topic
    representative_docs = topic_model.get_representative_docs(topic_id)
    if representative_docs:
        # Trova a quale paese/periodo corrisponde
        rep_meta = df[df["testo"] == representative_docs[0]].iloc[0]
        print(f"📌 Documento Rappresentativo: {rep_meta['paese']} ({rep_meta['periodo']})")
        print(f"📄 Estratto:\n{rep_meta['testo'][:450]}...")

## 7. Validazione Geografica (Esclusione Bias di Paese)

Verifichiamo che i cluster trovati riflettano effettivamente i driver delle crisi semantici e non siano un semplice raggruppamento per Paese (bias geografico).

Calcoliamo:
1. **Tabella di Contingenza (Paese vs. Topic)**.
2. **Entropia Geografica**: Valori alti indicano che il topic aggrega situazioni simili in paesi diversi.

In [ ]:
print("TABELLA DI CONTINGENZA (PAESE vs. TOPIC BERTopic):")
tabella_contingenza = pd.crosstab(df["paese"], df["topic"])
print(tabella_contingenza.to_string())

print(
    "\nENTROPIA GEOGRAFICA PER CIASCUN TOPIC:\n"
    "(Valori alti = il topic contiene molti paesi diversi, indicando coerenza semantica e basso bias geografico)"
)
for t_id in sorted(df["topic"].unique()):
    conteggio_paesi = df[df["topic"] == t_id]["paese"].value_counts()
    entropia_val = stats.entropy(conteggio_paesi)
    print(f"Topic {t_id}: Entropia Geografica = {entropia_val:.2f} (Paesi unici: {len(conteggio_paesi)})")

## 8. Visualizzazioni Native con BERTopic su Drive

Generiamo e salviamo le visualizzazioni interattive in formato HTML direttamente all'interno della cartella dedicata su Google Drive (`visualizzazioni_bertopic/`), rendendo semplice scaricarle e aprirle nel browser.

In [ ]:
# Crea la cartella per salvare le visualizzazioni HTML su Drive
os.makedirs(VISUALIZZAZIONI_DIR, exist_ok=True)
print(f"Le visualizzazioni HTML saranno salvate nella cartella Drive: {VISUALIZZAZIONI_DIR}")

In [ ]:
# 8.1 Mappa Interattiva delle Distanze tra Topic
fig_topics = topic_model.visualize_topics()
fig_topics.write_html(os.path.join(VISUALIZZAZIONI_DIR, "mappa_intertopica.html"))
fig_topics

In [ ]:
# 8.2 Proiezione 2D dei Documenti (Mappa dei Report)
fig_docs = topic_model.visualize_documents(
    df["testo"].tolist(),
    embeddings=embeddings,
    hide_annotations=True,
    custom_labels=False
)
fig_docs.write_html(os.path.join(VISUALIZZAZIONI_DIR, "mappa_report.html"))
fig_docs

In [ ]:
# 8.3 Distribuzione delle parole chiave per Topic (Bar Charts)
fig_barchart = topic_model.visualize_barchart(top_n_topics=10, n_words=8)
fig_barchart.write_html(os.path.join(VISUALIZZAZIONI_DIR, "barre_parole_chiave.html"))
fig_barchart

In [ ]:
# 8.4 Matrice di Somiglianza Termica (Heatmap)
fig_heatmap = topic_model.visualize_heatmap()
fig_heatmap.write_html(os.path.join(VISUALIZZAZIONI_DIR, "matrice_similitudine.html"))
fig_heatmap

In [ ]:
# 8.5 Dendrogramma Gerarchico
fig_hierarchy = topic_model.visualize_hierarchy()
fig_hierarchy.write_html(os.path.join(VISUALIZZAZIONI_DIR, "gerarchia_topic.html"))
fig_hierarchy

In [ ]:
print(f"Tutte le visualizzazioni sono state esportate con successo su Google Drive in:")
for f in os.listdir(VISUALIZZAZIONI_DIR):
    print(f"- {os.path.join(VISUALIZZAZIONI_DIR, f)}")